# DBLP ORCID Enrichment Pipeline

Downloads and processes ORCID and DBLP data, then runs enrichment benchmark and scoring via Spark.

In [ ]:
!pip install lxml

In [3]:
import os
import tarfile
import gzip
import xml.etree.ElementTree as ET
import json
import requests

# Configuration
WRKDIR = os.environ.get("WRKDIR", "./data")

ORCID_DATA = {
    "ORCID_2025_10_activities_0.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58809535", "type": "works"},
    "ORCID_2025_10_activities_1.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58809388", "type": "works"},
    "ORCID_2025_10_activities_2.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58809379", "type": "works"},
    "ORCID_2025_10_activities_3.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58809463", "type": "works"},
    "ORCID_2025_10_activities_4.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58809529", "type": "works"},
    "ORCID_2025_10_activities_5.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58809391", "type": "works"},
    "ORCID_2025_10_activities_6.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58830550", "type": "works"},
    "ORCID_2025_10_activities_7.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58834675", "type": "works"},
    "ORCID_2025_10_activities_8.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58834666", "type": "works"},
    "ORCID_2025_10_activities_9.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58834684", "type": "works"},
    "ORCID_2025_10_activities_X.tar.gz": {"url": "https://api.figshare.com/v2/file/download/58849081", "type": "works"},
    "ORCID_2025_10_summaries.tar.gz":    {"url": "https://api.figshare.com/v2/file/download/58834837", "type": "authors"}
}

DBLP_URL = "https://drops.dagstuhl.de/storage/artifacts/dblp/xml/2025/dblp-2025-11-01.xml.gz"
DBLP_DTD_URL = "https://drops.dagstuhl.de/storage/artifacts/dblp/xml/2023/dblp-2023-06-28.dtd"

# Derived paths
DBLP_DTD_NAME = DBLP_DTD_URL.split("/")[-1]
DBLP_DTD_FILE = os.path.join(WRKDIR, DBLP_DTD_NAME)
DBLP_XML_FILE = os.path.join(WRKDIR, "dblp.xml.gz")
DBLP_JSONL_FILE = os.path.join(WRKDIR, "dblp.jsonl")
OUTPUT_PATH = os.path.join(WRKDIR, "dblp_enriched")

os.makedirs(WRKDIR, exist_ok=True)

## Download and Process ORCID Data

In [25]:
from lxml import etree as ET
import json

NS = {
    "record": "http://www.orcid.org/ns/record",
    "common": "http://www.orcid.org/ns/common",
    "other-name": "http://www.orcid.org/ns/other-name",
    "person": "http://www.orcid.org/ns/person",
    "personal-details": "http://www.orcid.org/ns/personal-details",
    "work": "http://www.orcid.org/ns/work",
    "activities": "http://www.orcid.org/ns/activities",
}
_X = lambda e: ET.XPath(e, namespaces=NS)

_ORCID       = _X("string(.//common:orcid-identifier/common:path)")
_NAME        = _X(".//person:name")
_FAMILY      = _X("string(personal-details:family-name)")
_GIVEN       = _X("string(personal-details:given-names)")
_CREDIT      = _X("string(personal-details:credit-name)")
_OTHER_NAMES = _X(".//other-name:other-names/other-name:other-name/other-name:content/text()")
_WORK_EXT    = _X(".//work:work-summary//common:external-id")
_EXT_IDS     = _X(".//common:external-id")
_EXT_TYPE    = _X("string(common:external-id-type)")
_EXT_VALUE   = _X("string(common:external-id-value)")
_WORK_EL     = _X(".//work:work")
_WORK_TAG    = f"{{{NS['work']}}}work"

def parse_orcid_summary(xml_bytes):
    root = ET.fromstring(xml_bytes)
    name = _NAME(root)
    n = name[0] if name else None
    return {
        "orcid":      _ORCID(root) or None,
        "familyName": (_FAMILY(n) if n is not None else "") or None,
        "givenName":  (_GIVEN(n)  if n is not None else "") or None,
        "creditName": (_CREDIT(n) if n is not None else "") or None,
        "otherNames": [t for t in _OTHER_NAMES(root) if t],
    }

def parse_orcid_activity(xml_bytes):
    root = ET.fromstring(xml_bytes)
    if root.tag == _WORK_TAG:
        work_el = root
    else:
        found = _WORK_EL(root)
        work_el = found[0] if found else None
    if work_el is None:
        return None
    path = work_el.get("path")
    if not path:
        return "no path"
    return {
        "orcid": path.split("/")[1],
        "pids": [
            {
                "schema": _EXT_TYPE(e).strip() or None,
                "value":  _EXT_VALUE(e).strip() or None,
            }
            for e in _EXT_IDS(root)
        ],
    }

def handle_orcid_file(filename, xml_bytes):
    if "summaries" in filename:
        return json.dumps(parse_orcid_summary(xml_bytes))
    if "works" in filename:
        return json.dumps(parse_orcid_activity(xml_bytes))
    return None

In [ ]:
import os, tarfile, requests, bz2

for name, attr in ORCID_DATA.items():
    stem = os.path.join(WRKDIR, name[:-len(".tar.gz")])
    archive, arch_part = stem + ".tar.gz", stem + ".tar.gz.part"
    os.makedirs(os.path.join(WRKDIR, attr["type"]), exist_ok=True)
    stem = os.path.join(WRKDIR, attr["type"], name[:-len(".tar.gz")])
    out, out_part= stem + ".jsonl.bz2", stem + ".jsonl.bz2.part"

    if os.path.exists(out):
        print(f"[skip] {name}"); continue
    for stale in (arch_part, out_part):
        if os.path.exists(stale): os.remove(stale)


    if not os.path.exists(archive):
        print(f"[download] {name} from {attr["url"]}")
        with requests.get(attr["url"], allow_redirects=True, stream=True) as r, open(arch_part, "wb") as f:
            r.raise_for_status()
            for chunk in r.iter_content(chunk_size=8192):
                if chunk: f.write(chunk)
            f.flush(); os.fsync(f.fileno())
        os.rename(arch_part, archive)

    print(f"[process] {name}")
    with open(archive, "rb") as tmp, bz2.open(out_part, "wt", encoding="utf-8", compresslevel=3) as outf, \
         tarfile.open(fileobj=tmp, mode="r|gz") as tar:
        for member in tar:
            tar.members = []
            if not member.isfile(): continue
            ex = tar.extractfile(member)
            if ex is None: continue
            try: file_bytes = ex.read()
            finally: ex.close()
            try: jsonl = handle_orcid_file(member.name, file_bytes)
            except Exception as e: print(f"  skip {member.name}: {e}"); jsonl = None
            if jsonl: outf.write(jsonl if jsonl.endswith("\n") else jsonl + "\n")
        outf.flush()
    os.rename(out_part, out)
    os.remove(archive)
    print(f"[ok] {name}")

print("All archives processed.")

## Download and Process DBLP Data

In [4]:
from lxml import etree

def split_dblp_records(input_gz, dtd_file):
    """Iterate over top-level DBLP records from a gzipped XML dump."""
    with open(dtd_file, "rb") as f:
        dtd = etree.DTD(f)
    with gzip.open(input_gz, "rb") as f:
        context = etree.iterparse(
            f, events=("end",), tag=None, html=False,
            remove_blank_text=False, load_dtd=True,
            dtd_validation=False, attribute_defaults=False, no_network=True
        )
        for event, elem in context:
            if elem.getparent() is not None and elem.getparent().tag == "dblp":
                tree = etree.ElementTree(elem)
                if not dtd.validate(tree):
                    continue
                yield etree.tostring(elem, pretty_print=False, xml_declaration=True, encoding="UTF-8")
                elem.clear()
                while elem.getprevious() is not None:
                    del elem.getparent()[0]
        del context


def parse_dblp(line):
    """Parse a single DBLP XML record into a JSON string."""
    root = etree.fromstring(line.strip().replace("&", "").encode("utf-8"))
    ee = root.find("ee")
    if ee is not None and ee.text and "doi.org" in ee.text:
        doi = ee.text
        authors = root.findall("author")
        authors_list = [
            {"name": a.text, "orcid": a.attrib.get("orcid")}
            for a in authors
        ]
        return json.dumps({"doi": doi, "authors": authors_list})
    return None

In [ ]:
import os, bz2, requests

DBLP_SUBDIR = os.path.join(WRKDIR, "dblp")
os.makedirs(DBLP_SUBDIR, exist_ok=True)

out_name = os.path.basename(DBLP_JSONL_FILE)
if not out_name.endswith(".bz2"): out_name += ".bz2"
out = os.path.join(DBLP_SUBDIR, out_name)

dtd, dtd_part = DBLP_DTD_FILE, DBLP_DTD_FILE + ".part"
xml, xml_part = DBLP_XML_FILE, DBLP_XML_FILE + ".part"
out_part = out + ".part"

if os.path.exists(out):
    print(f"[skip] DBLP already processed ({out})")
else:
    for stale in (dtd_part, xml_part, out_part):
        if os.path.exists(stale): os.remove(stale)

    if not os.path.exists(dtd):
        print(f"[download] DTD -> {dtd}")
        with requests.get(DBLP_DTD_URL, stream=True) as r, open(dtd_part, "wb") as f:
            r.raise_for_status()
            for chunk in r.iter_content(chunk_size=8192):
                if chunk: f.write(chunk)
            f.flush(); os.fsync(f.fileno())
        os.rename(dtd_part, dtd)

    if not os.path.exists(xml):
        print(f"[download] XML -> {xml}")
        with requests.get(DBLP_URL, stream=True) as r, open(xml_part, "wb") as f:
            r.raise_for_status()
            for chunk in r.iter_content(chunk_size=8192):
                if chunk: f.write(chunk)
            f.flush(); os.fsync(f.fileno())
        os.rename(xml_part, xml)

    print("[process] DBLP dump")
    i = l = 0
    with bz2.open(out_part, "wt", encoding="utf-8", compresslevel=3) as fout:
        for item in split_dblp_records(xml, dtd):
            line = parse_dblp(item.decode("utf-8"))
            if line is not None:
                fout.write(line + "\n")
                l += 1
            i += 1
            if i % 100000 == 0:
                print(f"  processed {i} records, written {l} lines")
        fout.flush()
    os.rename(out_part, out)
    print(f"[ok] DBLP: {i} records, {l} lines -> {out}")

## Build Shaded JAR

In [ ]:
# Project paths - assumes notebook is run from notebook/ subdirectory
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
JAR = os.path.join(PROJECT_DIR, "target", "dblp-orcid-benchmark-0.1.1.jar")

In [ ]:
!cd {PROJECT_DIR} && mvn package -DskipTests -q
print(f"JAR: {JAR}")
assert os.path.exists(JAR), f"JAR not found at {JAR}"

## Run Benchmark

Runs  to enrich DBLP data with ORCID author matches.

In [ ]:
!spark-submit \
    --conf spark.driver.memory=8g \
    --class eu.openaire.dblp_benchmark.DBLPBenchmark \
    {JAR} \
    --orcidAuthorsPath ./data/authors \
    --orcidWorksPath ./data/works \
    --dblpDump ./data/dblp \
    --output ./data/benchmarks

## Produce false positive an false negative csv datasets


In [ ]:
!spark-submit \
    --class eu.openaire.dblp_benchmark.DBLPResultAnalyzer \
    {JAR} \
    --orcidAuthorsPath ./data/authors \
    --orcidWorksPath ./data/works \
    --cachePath ./data/benchmarks_joined_cache \
    --output ./data/results